# Parte A — Pretraining MAE-AST su FSD50K
### Replica di Baade et al. (Interspeech 2022) — Ablation masking ratio

**Obiettivo**: pretrainare MAE-AST su FSD50K con tre masking ratio (25%, 50%, 75%) e verificare il trend `acc_75 > acc_50 > acc_25`.

**Training loop**: PyTorch puro (fairseq non compatibile con Python 3.12 su Colab).

**Dataset**: FSD50K dev_audio (~40k clip a 44.1kHz), convertiti in .flac.

---

**Tempo stimato**: ~2-3 ore per masking ratio su A100.

In [ ]:
# CELLA 0 — Verifica GPU
import torch

if torch.cuda.is_available():
    nome_gpu = torch.cuda.get_device_name(0)
    vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {nome_gpu}  ({vram_gb:.1f} GB VRAM)')
else:
    raise SystemExit('GPU non trovata. Runtime -> Change runtime type -> A100 GPU')

DEVICE = torch.device('cuda')
print(f'Device: {DEVICE}')

In [ ]:
# CELLA 1 — Installa dipendenze
# NO fairseq — incompatibile con Python 3.12
!pip install soundfile librosa==0.10.1 timm==0.9.16 omegaconf==2.1.1 --quiet

import timm, librosa, torchaudio, soundfile
print(f'timm:       {timm.__version__}')
print(f'librosa:    {librosa.__version__}')
print(f'torchaudio: {torchaudio.__version__}')
print(f'torch:      {torch.__version__}')
print('Dipendenze pronte')

In [ ]:
# CELLA 2 — Clona la repo e monta Drive
import os, sys

REPO_URL = 'https://github.com/alessiopgg/deep_learning_project.git'
REPO_DIR = '/content/mae-ast-poggesi'

if os.path.exists(REPO_DIR):
    print('Repo gia presente, aggiorno...')
    os.system(f'cd {REPO_DIR} && git pull --quiet')
else:
    print('Clono la repo...')
    os.system(f'git clone {REPO_URL} {REPO_DIR} --quiet')

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print(f'Repo in: {REPO_DIR}')

from google.colab import drive
drive.mount('/content/drive')

RESULTS_DIR = '/content/drive/MyDrive/mae_ast_results'
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f'Risultati in: {RESULTS_DIR}')

In [ ]:
# CELLA 3 — Download e preparazione FSD50K
import subprocess

FSD50K_DIR  = '/content/FSD50K'
TSV_DIR     = '/content/fsd50k_tsv'
DEV_DIR     = os.path.join(FSD50K_DIR, 'FSD50K.dev_audio')

# Controlla se i TSV esistono gia
if os.path.exists(os.path.join(TSV_DIR, 'train.tsv')):
    with open(os.path.join(TSV_DIR, 'train.tsv')) as f:
        n_lines = sum(1 for _ in f) - 1  # escludi prima riga '/'
    print(f'TSV gia pronti: {n_lines} clip in train.tsv')
else:
    print('Preparo FSD50K...')
    # Lo script gestisce download, estrazione con 7z, conversione e generazione TSV
    !python {REPO_DIR}/scripts/prepare_fsd50k_tsv.py \
        --output_dir {TSV_DIR} \
        --audio_dir {FSD50K_DIR}

# Verifica
for split in ['train', 'valid']:
    tsv_path = os.path.join(TSV_DIR, f'{split}.tsv')
    with open(tsv_path) as f:
        lines = f.readlines()
    print(f'{split}.tsv: {len(lines)-1} clip (prima riga: "{lines[0].strip()}")')

In [ ]:
# CELLA 4 — Dataset per pretraining (standalone, NO fairseq)
import random
import numpy as np
import torch
import torch.nn as nn
import torchaudio
import torchaudio.compliance.kaldi as kaldi
from torch.utils.data import Dataset, DataLoader


class FSD50KPretrainDataset(Dataset):
    """
    Dataset per pretraining MAE-AST. Legge i TSV generati da prepare_fsd50k_tsv.py.
    Restituisce spettrogrammi fbank [1, 128, target_frames].
    """
    def __init__(self, tsv_path, target_sr=16000, n_mels=128, target_frames=512):
        self.target_sr = target_sr
        self.n_mels = n_mels
        self.target_frames = target_frames

        with open(tsv_path) as f:
            lines = f.readlines()
        # Prima riga e '/', le successive sono 'percorso\tn_campioni'
        self.entries = []
        for line in lines[1:]:
            parts = line.strip().split('\t')
            if len(parts) == 2:
                self.entries.append(parts[0])

        print(f'  FSD50KPretrainDataset: {len(self.entries)} clip da {tsv_path}')

    def __len__(self):
        return len(self.entries)

    def __getitem__(self, idx):
        path = self.entries[idx]
        try:
            waveform, sr = torchaudio.load(path)
        except Exception:
            # File corrotto: ritorna silenzio
            waveform = torch.zeros(1, self.target_sr * 5)
            sr = self.target_sr

        # Mono
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)

        # Resample a 16kHz
        if sr != self.target_sr:
            waveform = torchaudio.functional.resample(waveform, sr, self.target_sr)

        # Fbank 128-dim (stessa configurazione del paper)
        fbank = kaldi.fbank(
            waveform,
            sample_frequency=self.target_sr,
            num_mel_bins=self.n_mels,
            frame_length=25.0,
            frame_shift=10.0,
        )  # [T, 128]

        # Pad/truncate a target_frames
        T = fbank.shape[0]
        if T < self.target_frames:
            pad = torch.zeros(self.target_frames - T, self.n_mels)
            fbank = torch.cat([fbank, pad], dim=0)
        elif T > self.target_frames:
            # Random crop
            start = random.randint(0, T - self.target_frames)
            fbank = fbank[start:start + self.target_frames]

        # [target_frames, 128] -> [128, target_frames] -> [1, 128, target_frames]
        fbank = fbank.T.unsqueeze(0)
        return fbank


# Test rapido
ds_test = FSD50KPretrainDataset(os.path.join(TSV_DIR, 'train.tsv'))
sample = ds_test[0]
print(f'Shape campione: {sample.shape}  (atteso: [1, 128, 512])')

In [ ]:
# CELLA 5 — Modello MAE-AST standalone (NO fairseq)
#
# Reimplementa l'architettura MAE-AST usando solo PyTorch + timm.
# Architettura: Encoder (6 layer) + Decoder (2 layer) con masking.

import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class SimpleMAEAST(nn.Module):
    """
    MAE-AST semplificato per pretraining su Colab.
    Usa nn.TransformerEncoder di PyTorch invece del TransformerEncoder di fairseq.
    """
    def __init__(
        self,
        encoder_layers=6,
        decoder_layers=2,
        embed_dim=768,
        num_heads=12,
        ffn_dim=3072,
        dropout=0.1,
        kernel_size=16,
        mask_prob=0.75,
        mask_type='chunk',
    ):
        super().__init__()
        self.embed_dim = embed_dim
        self.kernel_size = kernel_size
        self.mask_prob = mask_prob
        self.mask_type = mask_type
        patch_dim = kernel_size * kernel_size  # 256

        # Patching
        self.batch_norm = nn.BatchNorm2d(1, affine=False)
        self.unfold = nn.Unfold(
            kernel_size=(kernel_size, kernel_size),
            stride=(kernel_size, kernel_size),
        )
        self.post_extract_proj = nn.Linear(patch_dim, embed_dim)

        # Mask embeddings
        self.encoder_mask_emb = nn.Parameter(torch.FloatTensor(embed_dim).uniform_())
        self.decoder_mask_emb = nn.Parameter(torch.FloatTensor(embed_dim).uniform_())

        # Encoder
        enc_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=num_heads, dim_feedforward=ffn_dim,
            dropout=dropout, activation='gelu', batch_first=True, norm_first=False,
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=encoder_layers)
        self.encoder_norm = nn.LayerNorm(embed_dim)

        # Decoder
        dec_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=num_heads, dim_feedforward=ffn_dim,
            dropout=dropout, activation='gelu', batch_first=True, norm_first=False,
        )
        self.decoder = nn.TransformerEncoder(dec_layer, num_layers=decoder_layers)
        self.decoder_norm = nn.LayerNorm(embed_dim)

        # Projection heads
        self.recon_proj = nn.Linear(embed_dim, patch_dim)
        self.class_proj = nn.Linear(embed_dim, patch_dim)

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def _create_mask(self, B, T):
        """Crea maschere per il masking. Restituisce (retained_idx, masked_idx)."""
        num_retained = max(1, int((1 - self.mask_prob) * T))
        num_masked = T - num_retained

        if self.mask_type == 'chunk':
            # Chunk masking (come SSAST/MAE-AST patch model)
            masked_set = set()
            freq_patches = 128 // self.kernel_size  # 8
            while len(masked_set) < num_masked:
                chunk_size = random.randrange(3, 6)
                start = random.randrange(T)
                for t_off in range(chunk_size):
                    for c_off in range(chunk_size):
                        idx = start + t_off + freq_patches * c_off
                        if idx < T:
                            masked_set.add(idx)
            masked_idx = list(masked_set)[:num_masked]
            retained_idx = list(set(range(T)) - set(masked_idx))
        else:
            # Random masking
            idx = list(range(T))
            random.shuffle(idx)
            retained_idx = idx[:num_retained]
            masked_idx = idx[num_retained:]

        return retained_idx, masked_idx

    def forward(self, source):
        """
        Args:
            source: [B, 1, 128, T_frames] spettrogramma fbank
        Returns:
            dict con logit_recon, logit_class, targets (tutti sui token mascherati)
        """
        B = source.shape[0]

        # BatchNorm + patching
        x = self.batch_norm(source) * 0.5
        patches = self.unfold(x).transpose(-1, -2)  # [B, n_patches, patch_dim]
        features = self.post_extract_proj(patches)   # [B, T, embed_dim]
        _, T, _ = features.shape

        # Masking (stessa maschera per tutto il batch per efficienza)
        retained_idx, masked_idx = self._create_mask(B, T)
        num_masked = len(masked_idx)

        # Applica maschera e passa solo token non mascherati all'encoder
        x_enc = features[:, retained_idx]  # [B, n_retained, embed_dim]

        # Encoder
        x_enc = self.encoder(x_enc)
        x_enc = self.encoder_norm(x_enc)

        # Ricostruisci sequenza completa per il decoder
        full_x = torch.empty_like(features)
        full_x[:, retained_idx] = x_enc
        full_x[:, masked_idx] = self.decoder_mask_emb

        # Decoder
        full_x = self.decoder(full_x)
        full_x = self.decoder_norm(full_x)

        # Estrai solo token mascherati per la loss
        x_masked = full_x[:, masked_idx]  # [B, num_masked, embed_dim]

        logit_recon = self.recon_proj(x_masked)  # [B, num_masked, patch_dim]
        logit_class = self.class_proj(x_masked)  # [B, num_masked, patch_dim]
        targets = patches[:, masked_idx]         # [B, num_masked, patch_dim]

        return {
            'logit_recon': logit_recon,
            'logit_class': logit_class,
            'targets': targets,
        }


def mae_ast_loss(output, recon_weight=10.0, class_weight=1.0):
    """Loss MAE-AST: MSE ricostruzione + InfoNCE classificazione."""
    logit_recon = output['logit_recon']
    logit_class = output['logit_class']
    targets = output['targets']

    # MSE reconstruction loss
    loss_recon = F.mse_loss(logit_recon, targets)

    # InfoNCE classification loss
    dots = torch.matmul(logit_class, targets.transpose(-1, -2))
    log_softmax = torch.log_softmax(dots, dim=-1)
    loss_nce = -torch.mean(torch.diagonal(log_softmax, dim1=-2, dim2=-1))

    loss = recon_weight * loss_recon + class_weight * loss_nce
    return loss, loss_recon.item(), loss_nce.item()


# Test rapido del modello
model_test = SimpleMAEAST(encoder_layers=2, decoder_layers=1).to(DEVICE)
x_test = torch.randn(2, 1, 128, 512).to(DEVICE)
out_test = model_test(x_test)
loss_test, _, _ = mae_ast_loss(out_test)
print(f'Test modello OK. Loss: {loss_test.item():.4f}')
print(f'  logit_recon: {out_test["logit_recon"].shape}')
print(f'  targets:     {out_test["targets"].shape}')
del model_test, x_test, out_test
torch.cuda.empty_cache()

In [ ]:
# CELLA 6 — Training loop PyTorch puro
import time
import yaml

# Carica configurazione
with open(f'{REPO_DIR}/config/pretrain.yaml') as f:
    pretrain_cfg = yaml.safe_load(f)

print('Configurazione pretraining:')
for section, params in pretrain_cfg.items():
    if isinstance(params, dict):
        for k, v in params.items():
            print(f'  {section}.{k}: {v}')


def pretrain_mae_ast(
    mask_prob,
    mask_type='chunk',
    tsv_dir=TSV_DIR,
    cfg=pretrain_cfg,
    device=DEVICE,
    save_dir=RESULTS_DIR,
):
    """Esegue il pretraining MAE-AST per un dato masking ratio."""
    print(f'\n{"="*60}')
    print(f'  PRETRAINING MAE-AST — mask_prob={mask_prob}')
    print(f'{"="*60}')

    # Dataset
    ds_train = FSD50KPretrainDataset(os.path.join(tsv_dir, 'train.tsv'))
    ds_valid = FSD50KPretrainDataset(os.path.join(tsv_dir, 'valid.tsv'))

    loader_train = DataLoader(
        ds_train, batch_size=24, shuffle=True,
        num_workers=2, pin_memory=True, drop_last=True,
    )
    loader_valid = DataLoader(
        ds_valid, batch_size=24, shuffle=False,
        num_workers=2, pin_memory=True,
    )

    # Modello
    model = SimpleMAEAST(
        encoder_layers=cfg['model']['encoder_layers'],
        decoder_layers=cfg['model']['decoder_layers'],
        embed_dim=cfg['model']['encoder_embed_dim'],
        num_heads=cfg['model']['encoder_attention_heads'],
        ffn_dim=cfg['model']['encoder_ffn_embed_dim'],
        dropout=cfg['model']['dropout'],
        mask_prob=mask_prob,
        mask_type=mask_type,
    ).to(device)

    n_params = sum(p.numel() for p in model.parameters()) / 1e6
    print(f'  Parametri: {n_params:.1f}M')

    # Ottimizzatore
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=cfg['optimizer']['lr'],
        betas=tuple(cfg['optimizer']['adam_betas']),
        eps=cfg['optimizer']['adam_eps'],
        weight_decay=cfg['optimizer']['weight_decay'],
    )

    recon_w = cfg['loss']['reconstruction_weight']

    # Calcola numero di epoche da max_update
    steps_per_epoch = len(loader_train)
    max_update = cfg['training']['max_update']
    n_epochs = max(1, max_update // steps_per_epoch)
    print(f'  Steps/epoca: {steps_per_epoch}')
    print(f'  Epoche: {n_epochs} (da max_update={max_update})')

    # Scheduler warmup + decay
    warmup_steps = cfg['optimizer']['warmup_updates']
    def lr_lambda(step):
        if step < warmup_steps:
            return step / max(1, warmup_steps)
        progress = (step - warmup_steps) / max(1, max_update - warmup_steps)
        return max(0.0, 1.0 - progress)
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

    # Training loop
    global_step = 0
    best_val_loss = float('inf')
    storia = {'train_loss': [], 'val_loss': [], 'loss_recon': [], 'loss_nce': []}

    for epoch in range(1, n_epochs + 1):
        model.train()
        epoch_loss = 0.0
        epoch_recon = 0.0
        epoch_nce = 0.0
        t_start = time.time()

        for batch_idx, fbank in enumerate(loader_train):
            fbank = fbank.to(device)  # [B, 1, 128, 512]

            output = model(fbank)
            loss, l_recon, l_nce = mae_ast_loss(output, recon_weight=recon_w)

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()

            epoch_loss += loss.item()
            epoch_recon += l_recon
            epoch_nce += l_nce
            global_step += 1

            if (batch_idx + 1) % 100 == 0:
                lr = optimizer.param_groups[0]['lr']
                print(f'  Epoch {epoch}/{n_epochs} | '
                      f'Step {global_step}/{max_update} | '
                      f'Loss: {loss.item():.4f} | '
                      f'LR: {lr:.2e}', end='\r')

            if global_step >= max_update:
                break

        n_batches = batch_idx + 1
        avg_loss = epoch_loss / n_batches
        avg_recon = epoch_recon / n_batches
        avg_nce = epoch_nce / n_batches
        t_epoch = time.time() - t_start

        storia['train_loss'].append(avg_loss)
        storia['loss_recon'].append(avg_recon)
        storia['loss_nce'].append(avg_nce)

        # Validation
        model.eval()
        val_loss = 0.0
        n_val = 0
        with torch.no_grad():
            for fbank in loader_valid:
                fbank = fbank.to(device)
                output = model(fbank)
                loss_v, _, _ = mae_ast_loss(output, recon_weight=recon_w)
                val_loss += loss_v.item()
                n_val += 1

        avg_val = val_loss / max(1, n_val)
        storia['val_loss'].append(avg_val)

        print(f'  Epoch {epoch:3d}/{n_epochs} | '
              f'Train: {avg_loss:.4f} (recon:{avg_recon:.4f} nce:{avg_nce:.4f}) | '
              f'Val: {avg_val:.4f} | '
              f'{t_epoch:.0f}s')

        # Salva checkpoint se migliora
        if avg_val < best_val_loss:
            best_val_loss = avg_val
            mask_pct = int(mask_prob * 100)
            ck_path = os.path.join(save_dir, f'mae_ast_pretrained_{mask_pct}pct.pt')
            torch.save({
                'model': model.state_dict(),
                'epoch': epoch,
                'global_step': global_step,
                'mask_prob': mask_prob,
                'val_loss': best_val_loss,
                'config': cfg,
            }, ck_path)
            print(f'  Checkpoint salvato: {ck_path}')

        if global_step >= max_update:
            print(f'  Raggiunto max_update={max_update}')
            break

    # Salva checkpoint finale
    mask_pct = int(mask_prob * 100)
    final_path = os.path.join(save_dir, f'mae_ast_pretrained_{mask_pct}pct_final.pt')
    torch.save({
        'model': model.state_dict(),
        'epoch': epoch,
        'global_step': global_step,
        'mask_prob': mask_prob,
        'val_loss': avg_val,
        'config': cfg,
    }, final_path)

    print(f'\n  Pretraining mask={mask_prob} completato.')
    print(f'  Best val loss: {best_val_loss:.4f}')
    print(f'  Checkpoint finale: {final_path}')

    del model, optimizer
    torch.cuda.empty_cache()

    return storia, final_path


print('Training loop pronto.')

In [ ]:
# CELLA 7 — Esegui pretraining per i tre masking ratio
import json

MASK_RATIOS = [0.25, 0.50, 0.75]
risultati_pretrain = {}

for mask_prob in MASK_RATIOS:
    storia, ck_path = pretrain_mae_ast(
        mask_prob=mask_prob,
        mask_type='chunk',
    )
    risultati_pretrain[f'mask_{int(mask_prob*100)}'] = {
        'storia': storia,
        'checkpoint': ck_path,
        'best_val_loss': min(storia['val_loss']),
    }

# Salva riepilogo
summary_path = os.path.join(RESULTS_DIR, 'partA_pretrain_summary.json')
summary = {}
for k, v in risultati_pretrain.items():
    summary[k] = {
        'best_val_loss': v['best_val_loss'],
        'checkpoint': v['checkpoint'],
        'train_loss_finale': v['storia']['train_loss'][-1],
    }
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)

print(f'\n{"="*60}')
print(f'  RIEPILOGO PRETRAINING')
print(f'{"="*60}')
for k, v in summary.items():
    print(f'  {k}: val_loss={v["best_val_loss"]:.4f}')
print(f'{"="*60}')
print(f'Salvato: {summary_path}')

In [ ]:
# CELLA 8 — Grafici loss di pretraining
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

colori = {'mask_25': '#E74C3C', 'mask_50': '#F39C12', 'mask_75': '#2E86C1'}

# Train loss
ax = axes[0]
for k, v in risultati_pretrain.items():
    ax.plot(v['storia']['train_loss'], label=k, color=colori[k])
ax.set_xlabel('Epoca')
ax.set_ylabel('Train Loss')
ax.set_title('Training Loss per masking ratio')
ax.legend()
ax.grid(alpha=0.3)

# Val loss
ax = axes[1]
for k, v in risultati_pretrain.items():
    ax.plot(v['storia']['val_loss'], label=k, color=colori[k])
ax.set_xlabel('Epoca')
ax.set_ylabel('Validation Loss')
ax.set_title('Validation Loss per masking ratio')
ax.legend()
ax.grid(alpha=0.3)

# Reconstruction vs NCE
ax = axes[2]
for k, v in risultati_pretrain.items():
    ax.plot(v['storia']['loss_recon'], label=f'{k} recon', color=colori[k], linestyle='-')
    ax.plot(v['storia']['loss_nce'], label=f'{k} nce', color=colori[k], linestyle='--')
ax.set_xlabel('Epoca')
ax.set_ylabel('Loss')
ax.set_title('Reconstruction vs InfoNCE')
ax.legend(fontsize=7)
ax.grid(alpha=0.3)

plt.tight_layout()
graph_path = os.path.join(RESULTS_DIR, 'partA_pretrain_loss.png')
plt.savefig(graph_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Grafico salvato: {graph_path}')